# E4 · Adaptación LoRA de CLIP sobre E3-100

**Objetivo.** Medir el beneficio adicional de adaptar el encoder visual después de E3.

**Pregunta.** ¿LoRA en la atención de CLIP mejora la recuperación frente a E3-100 sin perder transferencia a PetTrace?

**Hipótesis.** Una actualización de bajo rango puede representar señales de identidad que la MLP sobre embeddings congelados no captura.

| Elemento | Diseño fijado |
|---|---|
| Intervención | LoRA `r=8`, `alpha=16` en `q_proj` y `v_proj` de CLIP ViT-B/32 |
| Congelado | YOLO, pesos base/proyección de CLIP y cabeza E3-100 |
| Control | E3-100 semilla 42, fijado antes de observar E4 |
| Muestra | `pettrace-training@v4.100`; PetTrace no participa del entrenamiento |
| Selección | mAP macro por especie en validación pública; nunca por test |
| Evaluación | Test público v1 y PetTrace v4, con galerías separadas |
| Estado | **Pendiente hasta publicar ambos dominios en MLflow** |


### Entorno

En una T4 nueva, esta única instalación obtiene el código exacto desde PyPI. Las credenciales se leen de Colab Secrets y nunca se imprimen.

In [ ]:
%pip install -q --no-cache-dir "pettrace-ml[training,notebooks]==0.2.20"

In [ ]:
import json, os, shutil, sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import mlflow
import numpy as np
import torch
from IPython.display import Markdown, display
from google.colab import userdata
import pettrace
from pettrace.tracking import PRODUCTION_URI, source_identity

if not (3, 11) <= sys.version_info[:2] < (3, 14) or sys.platform != "linux":
    raise RuntimeError("E4 requires Colab Linux with Python 3.11–3.13")
if pettrace.__version__ != "0.2.20":
    raise RuntimeError("Install PetTrace 0.2.20 before continuing")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; a T4 is sufficient")
print("Access [1/2] Loading Colab Secrets…")
os.environ["MLFLOW_TRACKING_URI"] = PRODUCTION_URI
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise TypeError(f"Colab Secret {name} must be a non-empty text value")
    os.environ[name] = value
print("Access [2/2] Credentials available.")
EXPERIMENT, DATASET = "E4", "v4"
DATASET_COMMIT = "8b12db0dce41040a821f9792bea5b352342263ce"
TRAINING_DATASET_VERSION, SOURCE_SAMPLE_PCT, SEED = "v4.100", 100, 42
CLIP_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
BASE_MODEL = "models:/pettrace-embedder/2"
LOG_TO_MLFLOW = True
REGISTER_E4 = True
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
REPORT = Path("/content/reports") / "e4_v4" / RUN_LABEL
REPORT.mkdir(parents=True)
os.environ["PETTRACE_EXECUTION_ID"] = RUN_LABEL
SOURCE_IDENTITY = source_identity()
SOURCE_SHA256 = SOURCE_IDENTITY["source_sha256"]
mlflow.set_tracking_uri(PRODUCTION_URI); mlflow.set_registry_uri(PRODUCTION_URI)
print("Package:", pettrace.__version__, "| GPU:", torch.cuda.get_device_name(0), "| Report:", REPORT)

## Fase 1. Preparación y preprocesamiento de datos

E4 **carga** la versión refinada v4.100; no adquiere ZIP, no limpia y no ejecuta YOLO sobre el corpus. Reutiliza sus cajas verificadas para reproducir el recorte E2. Se conservan los splits públicos y el benchmark privado fijado.

In [ ]:
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
from pettrace.data import load
from pettrace.data.curation import sha256
from pettrace.data.volume import training_volume

series_path = Path("/content/drive/MyDrive/PetTrace/data-lookup/inventory100/volume-series.json")
series = json.loads(series_path.read_text())
receipt = series[str(SOURCE_SAMPLE_PCT)]
if receipt.get("version") != TRAINING_DATASET_VERSION:
    raise ValueError("The completed receipt does not contain v4.100")
print("Dataset [1/2] Restoring v4.100 and verifying hashes…")
prepared, training_source = load(TRAINING_DATASET_VERSION,
    cache_dir="/content/datasets/build/pettrace-training",
    drive_root="/content/drive/MyDrive/PetTrace")
if prepared.fingerprint != receipt.get("fingerprint"):
    raise ValueError("Prepared v4.100 fingerprint differs from its receipt")
manifest_rows = prepared.rows
screening = json.loads((prepared.root / "animal-screening.json").read_text())
for name in ("manifest.csv", "review.csv", "audit.json", "sources.json",
             "animal-screening.json", "prepared.json"):
    shutil.copy2(prepared.root / name, REPORT / name)
(REPORT / "volume-series.json").write_bytes(series_path.read_bytes())
coverage_metrics, coverage_params = training_volume(prepared, SOURCE_SAMPLE_PCT, REPORT)
print("Dataset [2/2] Prepared:", prepared.fingerprint, "| photos:", len(manifest_rows),
      "| identities:", prepared.summary["retained_identities"])

In [ ]:
from pettrace.tracking import production_context
from pettrace.experiments.e3 import load_benchmarks
production_context()
benchmarks = load_benchmarks(manifest_rows, REPORT, private_version=DATASET, private_commit=DATASET_COMMIT)
ds, public_ds, PUBLIC_REPORT = benchmarks.private, benchmarks.public, benchmarks.public_report
print("Public test:", len(public_ds.photos), "photos | PetTrace:", len(ds.photos), "photos")

## Fase 2. Representación y entrenamiento

Se descarga la cabeza E3-100 del run fijado y se verifica su SHA-256. CLIP usa la revisión inmutable indicada arriba. Solo los adaptadores LoRA son entrenables; antes de ajustar se comprueba que su inicialización reproduce E3.

LoRA sigue a [Hu et al. (2022)](https://arxiv.org/abs/2106.09685). La pérdida batch-hard sigue a [Hermans et al. (2017)](https://arxiv.org/abs/1703.07737). Capas, rango, presupuesto y control son decisiones de esta investigación.

In [ ]:
from pettrace.evaluation.core import load_model
from pettrace.experiments.e4 import load_control_head
from pettrace.training.lora import Config, PreparedPreprocessor, build_encoder, control_equivalence

base_model = load_model(BASE_MODEL)
if not base_model.golden_ok or base_model.pipeline.detector is None:
    raise RuntimeError("The fixed E2 base failed verification")
head, control_receipt = load_control_head(REPORT)
preprocessor = PreparedPreprocessor(base_model.pipeline, screening)
training_rows = [row for row in manifest_rows if row["split"] in {"train", "val"}]
train_rows = [row for row in training_rows if row["split"] == "train"]
gate = [next(row for row in sorted(train_rows, key=lambda item: item["sha256"]) if row["species"] == species)
        for species in ("cat", "dog")]
checked = preprocessor.validate(gate)
pixels = preprocessor.pixels(gate)
e2 = base_model.pipeline.session.run([base_model.pipeline.output_name],
    {base_model.pipeline.input_name: pixels})[0]
config = Config(seed=SEED)
encoder, adapted_layers = build_encoder(head, CLIP_REVISION, config=config)
parity = control_equivalence(encoder, pixels, e2)
(REPORT / "control-equivalence.json").write_text(json.dumps(
    {**parity, "cached_crop_photos": checked, "adapted_layers": adapted_layers}, indent=2))
print("Control parity:", parity, "| LoRA layers:", len(adapted_layers))

In [ ]:
from pettrace.data.checkpoints import DriveBackup
from pettrace.training.lora import fit, trainable_parameters

run_config = {"experiment": EXPERIMENT, "dataset": TRAINING_DATASET_VERSION,
              "dataset_fingerprint": prepared.fingerprint, "clip_revision": CLIP_REVISION,
              "control": control_receipt, "training": asdict(config),
              "trainable_parameters": trainable_parameters(encoder),
              "log_to_mlflow": LOG_TO_MLFLOW, "register_model": REGISTER_E4}
(REPORT / "training-config.json").write_text(json.dumps(run_config, indent=2))
report_backup = DriveBackup({"report": REPORT},
    Path("/content/drive/MyDrive/PetTrace/checkpoints/runs") / REPORT.name)
def checkpoint(record):
    report_backup(paths=[REPORT / "adapter.pt", REPORT / "history.json"])
    print(f"Epoch {record['epoch']:02d} | loss {record['train.loss']:.4f} | "
          f"validation macro mAP {record['val.macro_species_mAP']:.4f}", flush=True)
with report_backup:
    encoder, training_summary = fit(encoder, training_rows, preprocessor, REPORT,
                                    config=config, device="cuda", callback=checkpoint)
    (REPORT / "training-summary.json").write_text(json.dumps(training_summary, indent=2))
print("Selected epoch:", training_summary["best_epoch"],
      "| validation macro mAP:", training_summary["best_val_mAP"])

In [ ]:
from pettrace.training.lora import export_encoder
model_dir = REPORT / "model"
onnx_hashes = export_encoder(base_model.dir, encoder, model_dir,
    model_version="clip-vit-b32-crop-lora-e3-v4.100@v1", verification_pixels=pixels)
e4_model = load_model(str(model_dir))
if not e4_model.golden_ok:
    raise RuntimeError("Exported E4 model failed golden verification")
print("Exported:", onnx_hashes)

## Fase 3. Evaluación y análisis

El checkpoint ya quedó fijado por validación. Ahora se evalúan, por separado, el test público v1 y PetTrace v4. Rank-k usa el máximo score por identidad y mAP ordena fotografías en una galería cerrada de la misma especie. La EER es descriptiva porque se calcula sobre los mismos pares evaluados.

**Limitaciones.** Los benchmarks ya fueron observados en experimentos anteriores; no garantizan separación entre sesiones, búsqueda abierta ni generalización poblacional. Esta primera corrida usa una semilla y debe repetirse con 43/44 antes del cierre final.

In [ ]:
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import metrics_table, save
from pettrace.evaluation.research import save_benchmark_comparison
from pettrace.experiments.e4 import frozen_control
from pettrace.tracking import save_provenance

public_ev = evaluate(public_ds, e4_model.pipeline, zero_shot=True)
private_ev = evaluate(ds, e4_model.pipeline, zero_shot=True)
public_summary = save(public_ev, e4_model, str(PUBLIC_REPORT))
private_summary = save(private_ev, e4_model, str(REPORT))
save_provenance(PUBLIC_REPORT, e4_model); save_provenance(REPORT, e4_model)
public_e3, public_e3_summary = frozen_control(public_ev, "public", PUBLIC_REPORT)
private_e3, private_e3_summary = frozen_control(private_ev, "private", REPORT)
display(Markdown("### Test público\n" + metrics_table({"E3-100": public_e3_summary, "E4": public_summary})))
display(Markdown("### PetTrace v4\n" + metrics_table({"E3-100": private_e3_summary, "E4": private_summary})))
domain_table, _ = save_benchmark_comparison(public_ev, private_ev, REPORT, EXPERIMENT)
display(Markdown("### Dominios separados\n" + domain_table))
comparison = {domain: {metric: current["global"][metric] - control["global"][metric]
               for metric in ("rank1", "rank2", "mAP")}
              for domain, current, control in (("public", public_summary, public_e3_summary),
                                                ("private", private_summary, private_e3_summary))}
(REPORT / "comparison.json").write_text(json.dumps(comparison, indent=2))
print("E4 − E3:", comparison)

### Publicación obligatoria

La corrida solo termina cuando ambos dominios quedan publicados. Un error conserva `adapter.pt`, el modelo y los reportes en Drive; esta celda se puede reintentar sin volver a entrenar. El registro crea un candidato y no mueve aliases.

In [ ]:
from pettrace.experiments.e3 import publish_results, register_candidate
with report_backup:
    publication = publish_results(public_ev, private_ev, e4_model, REPORT, PUBLIC_REPORT, SOURCE_SHA256,
        experiment=EXPERIMENT, source_sample_pct=SOURCE_SAMPLE_PCT, config=config,
        training_summary=training_summary, prepared_fingerprint=prepared.fingerprint,
        training_source=training_source, coverage_metrics=coverage_metrics,
        coverage_params=coverage_params, private_dataset_version=DATASET)
run_id, public_run_id = publication["run_id"], publication["public_run_id"]
if REGISTER_E4:
    with report_backup:
        registration = register_candidate(REPORT, run_id, public_run_id, experiment=EXPERIMENT)
    print("Candidate version:", registration["version"], "| no alias promoted")
print("E4 complete · private:", run_id, "| public:", public_run_id)